# 📐 S&P 500 Pairs Trading 形成期 (Formation Period) 策略邏輯與公式詳解

## 📝 概述
在配對交易 (Pairs Trading) 中，**形成期 (Formation Period)**（預設 $F = 252$ 天）的核心任務是**篩選候選標的並建立具備統計套利價值的配對組合**。

> [!IMPORTANT]
> **本文件完全以 `strategies/formation/` 下實際運行的 `.py` 原始碼為準**進行解析，糾正了舊版 Notebook 文檔中的過時描述。

### 📂 策略檔案結構對照：
- **經典/進階 SSD 距離策略** $\rightarrow$ `strategies/formation/ssd_basic.py`, `strategies/formation/ssd.py`
- **純 DTW 與共整合 DTW 策略** $\rightarrow$ `strategies/formation/DTW_Pure_Notebook.py`, `strategies/formation/DTW_Cointegration_Paper.py`
- **HDBSCAN 密度分群系列策略** $\rightarrow$ `strategies/formation/HDBSCAN.py`, `HDBSCAN_CrossSector_UMAP.py`, `HDBSCAN_CrossSector_PCA.py`, `HDBSCAN_CrossSector_MultiFactor.py` 等

---

## 📏 一、 經典 SSD (Basic) 與進階 SSD (OLS) 形成期邏輯

在實際的系統程式碼中，**經典 SSD Basic 策略同樣套用了三關統計過濾以防止價差逆勢發散**。兩者在形成期的差異主要在於「價格正規化空間」與「對沖比例是否固定為 1.0」。

### 1.1 經典 SSD (Basic) 形成期邏輯 (`ssd_basic.py`)
- **價格幾何正規化 (Price Normalization)**：
  $$P'_{i, t} = \frac{P_{i, t}}{P_{i, 0}}$$
  以第一天價格 $P_{i,0}$ 為基準將股價歸一化為累積回報指數起點為 1.0。
- **幾何歐氏距離平方和 (SSD)**：
  $$\text{SSD}_{A,B} = \sum_{t=1}^F (P'_{A,t} - P'_{B,t})^2$$
  限制在同產業內計算。
- **避險比例**：固定為 1.0，即美元中性等市值對沖 ($v_a = 0.5, v_b = 0.5$)。
- **三關統計過濾 (已修正)**：程式碼在計算完 SSD 後，會挑選前 $\text{top\_n} \times 15$ 組候選配對進行慢速過濾：
  1. **ADF 共整合檢定** ($p < 0.05$)。
  2. **Ornstein-Uhlenbeck 半衰期過濾** ($2.0 \le \text{Half-Life} \le 40.0$ 天)。
  3. **Hurst 指數篩選** ($\text{Hurst} < 0.40$)。

### 1.2 進階 SSD (OLS) 形成期邏輯 (`ssd.py`)
- **對數價格 Z-Score 標準化**：
  $$P'_{i, t} = \frac{\ln(P_{i, t}) - \mu_{\ln(P_i)}}{\sigma_{\ln(P_i)}}$$
  將收益率波動度與價格規模完全歸一化。
- **對沖比例 (Hedge Ratio $\beta$)**：使用最小二乘法 (OLS) 計算兩者間的避險比例 $\beta$：
  $$\beta = \frac{\text{Cov}(P'_A, P'_B)}{\text{Var}(P'_B)}$$
  殘差為：$\epsilon_t = P'_{A, t} - \beta \cdot P'_{B, t}$。
- **三關統計過濾**：同樣套用上述 ADF ($p < 0.05$)、半衰期 ($2.0 \le HL \le 40.0$) 與 Hurst ($< 0.40$) 過濾篩選。

## ⏳ 二、 DTW (Dynamic Time Warping) 形成期邏輯

動態時間扭曲 (DTW) 能有效應對**兩隻股票在走勢上存在時間滯後 (Lag) 的非同步波動情況**。本系統實作了兩種 DTW 篩選邏輯：

### 2.1 純 DTW 距離策略 (`DTW_Pure_Notebook.py`)
- **累積總回報指數正規化**：先計算日收益率 $R_{i,t}$，再計算累積回報：
  $$P'_{i, t} = \prod_{\tau=1}^t (1 + R_{i, \tau})$$
- **無檢定篩選**：直接使用 `dtaidistance` (C優化庫) 計算同產業內所有股票的純 DTW 距離。**完全不套用**任何 ADF、Hurst 或半衰期等統計檢定，最大化保留潛在的扭曲相似配對。依 DTW 距離升序選擇 Top N。

### 2.2 許鈞翔 (2025) 論文對齊版共整合 DTW 策略 (`DTW_Cointegration_Paper.py`)
- **步驟 1：共整合與統計特徵預選**：
  對標準化對數價格進行雙向 OLS 回歸。只保留通過 **ADF 共整合檢定 ($p < 0.01$)**、半衰期在 $[2.0, 40.0]$ 天內、且 $\text{Hurst} < 0.40$ 的強均值回歸配對。
- **步驟 2：帶限制窗口的 Sakoe-Chiba DTW 距離**：
  為防止非理性時間扭曲，套用 Sakoe-Chiba 限制窗口 $W$ (預設為 15 天)，計算快速 DTW 距離，使時間對齊限制在合理的領先/落後天數內。
- **步驟 3：排序與 PCA 融合 (實驗組)**：
  系統支持兩種排序篩選模式：
  1. **`dtw` 模式 (對照組)**：依 DTW 距離升序排序選取前 $N$ 對。
  2. **`ssd_dtw_pca` 模式 (實驗組)**：將 SSD 與 DTW 距離標準化後，利用主成分分析 (PCA) 提取第一主成分 PC1 得分作為「綜合距離指標」並升序排序，融合了幾何形狀相似性 (SSD) 與時序對齊特徵 (DTW)。

In [ ]:
# DTW_Cointegration_Paper.py 限制窗口的 Sakoe-Chiba DTW 與 PCA 融合篩選
import numpy as np
from sklearn.decomposition import PCA

def compute_sakoe_chiba_dtw(x, y, window=15):
    n, m = len(x), len(y)
    dp = np.full((n + 1, m + 1), np.inf)
    dp[0, 0] = 0.0
    
    for i in range(1, n + 1):
        start_j = max(1, i - window)
        end_j = min(m, i + window)
        for j in range(start_j, end_j + 1):
            cost = (x[i - 1] - y[j - 1]) ** 2
            dp[i, j] = cost + min(
                dp[i - 1, j],     # 插入
                dp[i, j - 1],     # 刪除
                dp[i - 1, j - 1]  # 匹配
            )
    return float(dp[n, m])

def fuse_ssd_dtw_via_pca(ssd_list, dtw_list):
    # 將 SSD 與 DTW 距離組合成二維特徵矩陣並標準化
    data = np.column_stack([ssd_list, dtw_list])
    mean = np.mean(data, axis=0)
    std = np.std(data, axis=0) + 1e-12
    data_scaled = (data - mean) / std
    
    # 進行 PCA 降維，取得第一主成分 PC1得分
    pca = PCA(n_components=1)
    pc1_score = pca.fit_transform(data_scaled).squeeze()
    
    # 確保方向一致：若與原始距離呈負相關，則反轉得分
    if pca.components_[0, 0] < 0:
        pc1_score = -pc1_score
    return pc1_score

## 🌐 三、 HDBSCAN 系列密度分群形成期邏輯

HDBSCAN 系列策略將股票映射到**特徵空間**進行密度分群，再於同分群內進行配對，排除無規律的隨機噪聲標的。此處區分為「產業內分群」與「全市場跨產業分群」兩大邏輯類型：

### 3.1 產業內 HDBSCAN 密度分群 (`HDBSCAN.py`)
- **特徵空間 (13維)**：對每檔股票收益率序列提取 13 維時序與統計特徵。
- **分群流程**：只在**同產業內**使用 **UMAP** (或 PCA) 降維至 5 維，並透過 HDBSCAN 分群，排除噪音點 ($label = -1$)。配對僅在「同產業且同分群」的股票中產生，進一步進行雙向 OLS 與 Engle-Granger ADF 共整合、半衰期與 Hurst 過濾。

### 3.2 全市場跨產業 HDBSCAN 聚類策略 (`HDBSCAN_CrossSector_UMAP.py` / `PCA.py` / `MultiFactor.py`)
這是本量化平台的重要進階修正。為克服「同產業配對池過小」且尋找「跨行業的替代性統計關係」，跨產業策略做出了以下優化：
- **全市場聚類與跨產業配對 (Cross-Sector)**：
  降維與 HDBSCAN 分群是**在全市場（而非各產業內部）**股票上執行。只要被劃分到同一個聚類群落（$label \ne -1$）的股票即可兩兩配對，即使它們來自**不同的 GICS 板塊**（例如科技與金融）。
- **多維因子特徵空間**：
  - `HDBSCAN_CrossSector_UMAP.py` 與 `_PCA.py` 使用 13 維時序特徵降維聚類。
  - `HDBSCAN_CrossSector_MultiFactor.py` 使用 **6 大金融穩健因子**（市場 Beta、波動率、偏態、峰態、長期價格趨勢斜率、特異波動率），並且**跳過降維步驟**直接進行 HDBSCAN 跨產業分群，以保留最純粹的金融意義。
- **更嚴格的 5 道篩選關卡**：
  為了防範跨產業隨機配對導致的假性共整合 (Spurious Cointegration)，套用了高達 5 重統計檢定：
  1. **皮爾森相關係數 (Correlation Filter)**：要求對數價格之 $\text{Correlation} \ge 0.50$。
  2. **ADF 共整合檢定**：要求 $p \text{-value} < \text{adf\_pvalue\_threshold}$ (預設 0.01 或 0.05)。
  3. **O-U 半衰期**：價差均值回歸半衰期必須滿足 $\text{halflife\_min} \le HL \le \text{halflife\_max}$ (預設 $2.0 \le HL \le 63.0$ 天)。
  4. **Hurst 指數**：要求 $\text{Hurst} < \text{hurst\_threshold}$ (預設 0.5)。
  5. **均值交叉次數 (Zero Crossings Filter)**：形成期殘差序列跨越其均值的次數必須滿足 $\text{Zero\_Crossings} \ge \text{min\_zero\_crossings}$ (預設 5次)，確認價差回歸的頻繁度。
- **Han et al. 2021：Mom1 截面動量差篩選**：
  計算個股一個月動量差 $\text{Mom1} = \ln(P_t) - \ln(P_{t-21})$，計算兩者的動量差值 $\text{Mom1\_Diff} = |\text{Mom1}_A - \text{Mom1}_B|$。差值越大代表近期發生非對稱過度偏離，具有更強的反轉獲利空間。

In [ ]:
# HDBSCAN_CrossSector_UMAP.py 跨產業配對與 5 道篩選核心邏輯
import numpy as np

def filter_cross_sector_pairs(log_a, log_b, min_corr=0.50, min_zero_crossings=5):
    # 1. Pearson Correlation Filter
    corr = np.corrcoef(log_a, log_b)[0, 1]
    if corr < min_corr:
        return False, corr, 0
        
    # 2. OLS fitting (A on B) & residual calculation
    n_len = len(log_a)
    x_mat = np.column_stack([np.ones(n_len), log_b])
    coeffs = np.linalg.lstsq(x_mat, log_a, rcond=None)[0]
    resid = log_a - coeffs[0] - coeffs[1] * log_b
    
    # 3. Zero Crossings Filter (穿越均值次數檢測)
    mean_val = np.mean(resid)
    demeaned = resid - mean_val
    # 計算符號變更的次數
    zero_crossings = np.sum(np.diff(np.sign(demeaned)) != 0)
    if zero_crossings < min_zero_crossings:
        return False, corr, zero_crossings
        
    return True, corr, zero_crossings

## 📊 四、 所有策略形成期特徵與配對標準對比

| 策略特徵 | 經典 SSD (Basic) | 進階 SSD (OLS) | 純 DTW 距離 | 論文共整合 DTW | HDBSCAN 產業內聚類 | HDBSCAN 跨產業聚類 |
| :--- | :--- | :--- | :--- | :--- | :--- | :--- |
| **跨產業配對** | 否 (限定同產業) | 否 (限定同產業) | 否 (限定同產業) | 否 (限定同產業) | 否 (限定同產業) | **是 (全市場跨板塊聚類)** |
| **正規化方法** | $P_{i,t}/P_{i,0}$ | 對數價格 Z-Score | 累積收益指數 | 對數價格 Z-Score | 對數價格 Z-Score | 對數價格 Z-Score |
| **特徵維度** | 無 | 無 | 無 | 無 | 13維時序統計特徵 | 13維時序 或 6維金融因子 |
| **降維方法** | 無 | 無 | 無 | 無 | UMAP (5維) | UMAP/PCA 或 無 (多因子) |
| **過濾檢定** | ADF, 半衰期, Hurst | ADF, 半衰期, Hurst | 無 | ADF, 半衰期, Hurst | ADF, 半衰期, Hurst | **ADF, 半衰期, Hurst, 相關性, 均值穿越次數** |
| **動量差過濾** | 否 | 否 | 否 | 否 | 否 | **是 (Mom1 截面動量差)** |
| **避險比例 $\beta$** | 固定為 1.0 | 形成期 OLS 斜率 | 固定為 1.0 | 形成期 OLS 斜率 | 形成期 OLS 斜率 | 形成期 OLS 斜率 |

---